## Initialization

In [ ]:
# Importing needed code

import re
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    TypeVar,
    Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log

import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from pint import Quantity
from scipy.stats import linregress

from data_processing.arc_paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
from data_processing.processing.bimodal_fitting import (
    get_psd_energy_histogram,
    scan_histogram_slices,
    find_failed_slices,
    BimodalBounds,
    BimodalParams
)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import classify
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing.helpers import (
    stop, get_input_with_default, input_experiment_ids, get_midpoints_from_min_max_series)
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)

In [ ]:
def load_rate_data(experiment_name: str, time_bin_length: int) -> pd.DataFrame:
    file_name = f"{experiment_name}_data_{time_bin_length}s_bin.csv"
    output_root = get_report_root(experiment_name)
    file_path = output_root / file_name
    df = pd.read_csv(file_path)
    return df

In [ ]:
T = TypeVar("T")


def input_with_validation(
    converter: Callable[[Any], T],
    prompt: str,
    invalid_msg: str
) -> T:
    output = None
    while output is None:
        in_val = input(prompt)
        try:
            output = converter(in_val)
        except ValueError:
            print(invalid_msg)
    return output


def input_yes_no(
    prompt: str,
    default_yes: bool = True
) -> bool:
    possible_yes = ['y', 'yes']
    possible_no = ['n', 'no']
    full_prompt = f"{prompt} (y/n)\nPress Enter for {'yes' if default_yes else 'no'}"

    in_val = input(full_prompt)
    if in_val.lower() in possible_yes:
        return True
    elif in_val.lower() in possible_no:
        return False
    else:
        return default_yes

In [ ]:
StableInterval = tuple[float, float]  # in ps after exp. start
StableIntervals = list[StableInterval]


def find_stable_region(
    rate_df: pd.DataFrame,
    duration: float | None = None
) -> StableInterval:
    # TODO let user enter stable regions
    # check stability via linear fit
    # show slope to user
    # let user confirm stable region, or re-enter region
    done = False
    # start = None
    while not done:
        end = None
        start = input_with_validation(
            float,
            "Enter start time (in minutes after experiment start): ",
            "Not a valid number, try again"
        )
        # while start is None:
        #     in_val = input("Enter start time (in minutes after experiment start): ")
        #     try:
        #         start = float(in_val)
        #     except ValueError:
        #         print("Not a valid number, try again")
        if duration is not None:
            end = start + duration
        if end is None:
            end = input_with_validation(
                float,
                "Enter end time (in minutes after experiment start): ",
                "Not a valid number, try again"
            )
            # in_val = input("Enter start time (in minutes after experiment start): ")
            # try:
            #     end = float(in_val)
            # except ValueError:
            #     print("Not a valid number, try again")
        if end <= start:
            print("End must be greater than start, try again")
            # start = None
            # end = None
            continue

        x_all = rate_df["Bin time (s)"]
        y_all = rate_df["Neutron rate (cps)"]
        duration_series_idx = x_all.between(start * 60, end * 60)
        x = x_all[duration_series_idx]
        y = y_all[duration_series_idx]
        result = linregress(x, y)
        print(f"Rate slope = {result.slope:.2E} cps/s +/- {result.stderr:.2E}")
        in_done = get_input_with_default(
            """\
Is this region stable?
Enter y/n, or press Enter for no
""",
            "n",
            str
        )
        done = in_done != "n"
    return start, end


def find_stable_regions(
    rate_df: pd.DataFrame,
    count: int = 3,
    persist_duration: bool = True,
) -> StableIntervals:
    if count < 1:
        raise ValueError("count must be 1 or greater")

    regions = []

    print("---Region 1---")
    bg_region = find_stable_region(rate_df)
    regions.append(bg_region)
    bg_duration = None
    if persist_duration:
        bg_start, bg_end = bg_region
        bg_duration = bg_end - bg_start

    for i in range(count - 1):
        print(f"---Region {i+2}---")
        region = find_stable_region(rate_df, bg_duration)
        regions.append(region)

    return regions

## Input Settings

In [ ]:
print("Enter beam-loading experiment IDs")
beam_experiment_ids = input_experiment_ids()
print()
print("Enter E-cell experiment IDs")
ecell_experiment_ids = input_experiment_ids()
all_experiment_ids = list(zip(beam_experiment_ids, ecell_experiment_ids))

In [ ]:
time_bin_length = get_input_with_default(
    """\
Enter bin length (in seconds)
Press Enter for default (30)
""",
    30,
    int
)

## Data Loading

In [ ]:
LinkableDataset = dict[str, dict[ExperimentDataKey, Any] | str]
LinkedDatasets = tuple[LinkableDataset, LinkableDataset]
all_experiment_data: list[LinkedDatasets] = [
    (
        {"exp_id": beam_exp_id, "data": {}},
        {"exp_id": ecell_exp_id, "data": {}}
    )
    for beam_exp_id, ecell_exp_id in all_experiment_ids
]

In [ ]:
for linked_dataset in all_experiment_data:
    beam_dataset, ecell_dataset = linked_dataset
    beam_id = beam_dataset['exp_id']
    beam_data = beam_dataset['data']
    ecell_id = ecell_dataset['exp_id']
    ecell_data = ecell_dataset['data']
    for exp_id, exp_data in [(beam_id, beam_data), (ecell_id, ecell_data)]:
        detector_df = load_psd(exp_id)
        rate_df = load_rate_data(exp_id, time_bin_length)
        exp_data[ExperimentDataKey.UNCLASSIFIED] = detector_df
        exp_data[ExperimentDataKey.ALL_BINNED_DATA] = rate_df

## Analysis

In [ ]:
bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
n_error_col_name = BinningDataframeColumn.NEUTRON_RATE_ERROR.value
dot_size = 8

for linked_dataset in all_experiment_data:
    beam_dataset, ecell_dataset = linked_dataset
    beam_id = beam_dataset['exp_id']
    beam_data = beam_dataset['data']
    beam_label = f"Beam ({beam_id})"
    ecell_id = ecell_dataset['exp_id']
    ecell_data = ecell_dataset['data']
    ecell_label = f"Ecell ({ecell_id})"

    beam_neutrons = beam_data[ExperimentDataKey.ALL_BINNED_DATA]
    ecell_neutrons = ecell_data[ExperimentDataKey.ALL_BINNED_DATA]

    fig, ax = plt.subplots(figsize=(12, 8), dpi=300)

    for label, binned_neutrons in [
        (beam_label, beam_neutrons),
        (ecell_label, ecell_neutrons)
    ]:
        zeroed_bins = binned_neutrons[bin_time_col_name] / 60
        rates = binned_neutrons[n_rate_col_name]
        rate_errors = binned_neutrons[n_error_col_name]
        ax.errorbar(
            zeroed_bins,
            rates,
            yerr=rate_errors,
            fmt=".",
            linestyle='',
            markersize=dot_size,
            capsize=dot_size,
            alpha=0.2,
            label=label            
        )
    ax.grid(which="both")
    ax.set_xlabel("Time [minutes]", fontsize=14)  # Update x-axis label
    ax.set_ylabel("Neutron count rate [1/s]", fontsize=14)
    # ax.set_ylim(0, 22)
    # ax.set_ylim(3.5, 5.0)
    ax.tick_params(labelsize=12)
    ax.xaxis.set_minor_locator(plt.MultipleLocator(5))

    # Add a title above the plot
    fig.text(
        0.5,
        0.90,
        f"{exp_id} neutron count rate over time (Dwell time {time_bin_length}s)",
        ha='center',
        fontsize=20
    )
    fig.legend()

    # Show the plot (optional)
    plt.show()

In [ ]:
# let user enter stable regions
# check stability via linear fit
# show slope to user
# let user confirm stable region, or re-enter region

for linked_dataset in all_experiment_data:
    beam_dataset, ecell_dataset = linked_dataset
    beam_id = beam_dataset['exp_id']
    beam_data = beam_dataset['data']
    ecell_id = ecell_dataset['exp_id']
    ecell_data = ecell_dataset['data']

    print(f"-----{beam_id}/{ecell_id}-----")
    persist_duration = input_yes_no(
        "Keep the same duration for all regions?", default_yes=False
    )
    final_background = input_yes_no(
        "Use a final background region?"
    )
    region_count = 3 if final_background else 2

    # event_counts = {"Beam": [], "Ecell": []}
    # durations = {"Beam": [], "Ecell": []}

    for exp_type, exp_id, exp_data in [
        ("Beam", beam_id, beam_data),
        ("Ecell", ecell_id, ecell_data)
    ]:
        event_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
        rate_df = exp_data[ExperimentDataKey.ALL_BINNED_DATA]

        print(f"---{exp_type}: {exp_id}---")
        stable_regions = find_stable_regions(
            rate_df,
            count=region_count,
            persist_duration=persist_duration,
        )
        print()
        exp_data['stable_regions'] = stable_regions
    print()

    #     print(f"-----Experiment {exp_id}-----")
    #     print("---Stable Regions---")
    #     for i, region in enumerate(stable_regions):
    #         start, end = region
    #         duration = end - start
    #         print(f"Region {i+1}: {start:.1f} min to {end:.1f} min")
    #         print(f"Duration: {duration:.1f} min")

    #         time_col = event_df['TIMETAG']
    #         # count = event_df[event_df[
    #         #     (start * 1e12) < time_col & time_col <= (end * 1e12)
    #         # ]].shape[0]
    #         is_in_region = time_col.between(start * 60 * 1e12, end * 60 * 1e12)
    #         events_in_region = event_df[
    #             is_in_region
    #         ]
    #         event_count = events_in_region.shape[0]
    #         durations[exp_type].append(duration)
    #         event_counts[exp_type].append(event_count)

    # beam_counts = event_counts["Beam"]
    # beam_durations = durations["Beam"]
    # ecell_counts = event_counts["Ecell"]
    # ecell_durations = durations["Ecell"]
    # if region_count == 3:
    #     beam_bg_count, beam_count, final_beam_bg_count = beam_counts
    #     beam_bg_count += final_beam_bg_count
    #     beam_bg_duration, beam_duration, final_beam_bg_duration = beam_durations
    #     beam_bg_duration += final_beam_bg_duration
    #     ecell_bg_count, ecell_count, final_ecell_bg_count = ecell_counts
    #     ecell_bg_count += final_ecell_bg_count
    #     ecell_bg_duration, ecell_duration, final_ecell_bg_duration = ecell_durations
    #     ecell_bg_duration += final_ecell_bg_duration
    # else:
    #     beam_bg_count, beam_count = beam_counts
    #     beam_bg_duration, beam_duration = beam_durations
    #     ecell_bg_count, ecell_count = ecell_counts
    #     ecell_bg_duration, ecell_duration = ecell_durations

    # print(f"-----{beam_id}/{ecell_id}-----")
    # print("---Counts---")
    # print(f"Beam Background:  {beam_bg_count}")
    # print(f"ECell Background: {ecell_bg_count}")
    # print(f"Beam Loading:     {beam_count}")
    # print(f"Ecell:            {ecell_count}")

    # # print("---Durations---")
    # # print(f"Background:   {bg_duration}")
    # # print(f"Beam Loading: {beam_duration}")
    # # print(f"Ecell:        {ecell_duration}")

    # beam_bg_per_min = beam_bg_count / beam_bg_duration
    # ecell_bg_per_min = ecell_bg_count / ecell_bg_duration
    # beam_per_min = beam_count / beam_duration
    # ecell_per_min = ecell_count / ecell_duration
    # print("---Counts (per minute)---")
    # print(f"Beam Background:  {beam_bg_per_min:.2f}")
    # print(f"ECell Background: {ecell_bg_per_min:.2f}")
    # print(f"Beam Loading:     {beam_per_min:.2f}")
    # print(f"Ecell:            {ecell_per_min:.2f}")

    # bg_delta = abs(ecell_bg_per_min - beam_bg_per_min)
    # bg_delta_rel = bg_delta / min(beam_bg_per_min, ecell_bg_per_min)
    # print("---Background Comparison---")
    # print(f"Difference: {bg_delta:.2f}")
    # print(f"Percent:    {bg_delta_rel:.2%}")

    # beam_no_bg = beam_per_min - beam_bg_per_min
    # ecell_no_bg = ecell_per_min - ecell_bg_per_min
    # print("---Relative to Background---")
    # print(f"Beam Loading: {beam_no_bg:.2f}")
    # print(f"Ecell:        {ecell_no_bg:.2f}")

    # ecell_rel_inc_no_bg = (ecell_no_bg - beam_no_bg) / beam_no_bg
    # ecell_rel_inc = (ecell_per_min - beam_per_min) / beam_per_min
    # print()
    # print(f"Percent Increase (BG removed): {ecell_rel_inc_no_bg:.2%}")
    # print(f"Percent Increase (with BG): {ecell_rel_inc:.2%}")

In [ ]:
all_experiment_data

In [ ]:
# TODO display results
for linked_dataset in all_experiment_data:
    beam_dataset, ecell_dataset = linked_dataset
    beam_id = beam_dataset['exp_id']
    beam_data = beam_dataset['data']
    beam_regions = beam_data['stable_regions']
    ecell_id = ecell_dataset['exp_id']
    ecell_data = ecell_dataset['data']
    ecell_regions = ecell_data['stable_regions']

    # persist_duration = input_yes_no(
    #     "Keep the same duration for all regions?", default_yes=False
    # )
    # final_background = input_yes_no(
    #     "Use a final background region?"
    # )
    # region_count = 3 if final_background else 2

    event_counts = {"Beam": [], "Ecell": []}
    durations = {"Beam": [], "Ecell": []}

    for exp_type, exp_id, exp_data, stable_regions in [
        ("Beam", beam_id, beam_data, beam_regions),
        ("Ecell", ecell_id, ecell_data, ecell_regions)
    ]:
        event_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
        rate_df = exp_data[ExperimentDataKey.ALL_BINNED_DATA]

    #     print(f"---{exp_type}: {exp_id}---")
    #     stable_regions = find_stable_regions(
    #         rate_df,
    #         count=region_count,
    #         persist_duration=persist_duration,
    #     )
    #     exp_data['stable_regions'] = stable_regions

        print(f"-----Experiment {exp_id}-----")
        print("---Stable Regions---")
        for i, region in enumerate(stable_regions):
            start, end = region
            duration = end - start
            print(f"Region {i+1}: {start:.1f} min to {end:.1f} min")
            print(f"Duration: {duration:.1f} min")

            time_col = event_df['TIMETAG']
            # count = event_df[event_df[
            #     (start * 1e12) < time_col & time_col <= (end * 1e12)
            # ]].shape[0]
            is_in_region = time_col.between(start * 60 * 1e12, end * 60 * 1e12)
            events_in_region = event_df[
                is_in_region
            ]
            event_count = events_in_region.shape[0]
            durations[exp_type].append(duration)
            event_counts[exp_type].append(event_count)

    beam_counts = event_counts["Beam"]
    beam_durations = durations["Beam"]
    ecell_counts = event_counts["Ecell"]
    ecell_durations = durations["Ecell"]

    if len(beam_regions) == 3 and len(ecell_regions) == 3:
        beam_bg_count, beam_count, final_beam_bg_count = beam_counts
        beam_bg_count += final_beam_bg_count
        beam_bg_duration, beam_duration, final_beam_bg_duration = beam_durations
        beam_bg_duration += final_beam_bg_duration
        ecell_bg_count, ecell_count, final_ecell_bg_count = ecell_counts
        ecell_bg_count += final_ecell_bg_count
        ecell_bg_duration, ecell_duration, final_ecell_bg_duration = ecell_durations
        ecell_bg_duration += final_ecell_bg_duration
    else:
        beam_bg_count, beam_count = beam_counts
        beam_bg_duration, beam_duration = beam_durations
        ecell_bg_count, ecell_count = ecell_counts
        ecell_bg_duration, ecell_duration = ecell_durations

    print(f"-----{beam_id}/{ecell_id}-----")
    print("---Counts---")
    print(f"Beam Background:  {beam_bg_count}")
    print(f"ECell Background: {ecell_bg_count}")
    print(f"Beam Loading:     {beam_count}")
    print(f"Ecell:            {ecell_count}")

    # print("---Durations---")
    # print(f"Background:   {bg_duration}")
    # print(f"Beam Loading: {beam_duration}")
    # print(f"Ecell:        {ecell_duration}")

    beam_bg_per_min = beam_bg_count / beam_bg_duration
    ecell_bg_per_min = ecell_bg_count / ecell_bg_duration
    beam_per_min = beam_count / beam_duration
    ecell_per_min = ecell_count / ecell_duration
    print("---Counts (per minute)---")
    print(f"Beam Background:  {beam_bg_per_min:.2f}")
    print(f"ECell Background: {ecell_bg_per_min:.2f}")
    print(f"Beam Loading:     {beam_per_min:.2f}")
    print(f"Ecell:            {ecell_per_min:.2f}")

    bg_delta = abs(ecell_bg_per_min - beam_bg_per_min)
    bg_delta_rel = bg_delta / min(beam_bg_per_min, ecell_bg_per_min)
    print("---Background Comparison---")
    print(f"Difference: {bg_delta:.2f}")
    print(f"Percent:    {bg_delta_rel:.2%}")

    beam_no_bg = beam_per_min - beam_bg_per_min
    ecell_no_bg = ecell_per_min - ecell_bg_per_min
    print("---Relative to Background---")
    print(f"Beam Loading: {beam_no_bg:.2f}")
    print(f"Ecell:        {ecell_no_bg:.2f}")

    ecell_rel_inc_no_bg = (ecell_no_bg - beam_no_bg) / beam_no_bg
    ecell_rel_inc = (ecell_per_min - beam_per_min) / beam_per_min
    print()
    print(f"Percent Increase (BG removed): {ecell_rel_inc_no_bg:.2%}")
    print(f"Percent Increase (with BG): {ecell_rel_inc:.2%}")